# 삼양패키징 매출 × 경제지표 상관관계 분석

DART 공시 원문(`RAG/data/dart_xml/`)에서 삼양패키징의 매출액을 직접 파싱해서 추출하고,
`Steam_Sales` 대시보드가 이미 수집해 둔 경제지표(PostgreSQL/Neon)와 대조한다.

삼양패키징은 PET용기·아셉틱 음료용기를 만드는 회사라(음료·생수·주류 업체향 납품),
원자재는 석유화학 기반 수지(PET)다. 그래서 `samyang_food_correlation.ipynb`가 봤던
곡물/축산물 지표 대신 **유가·환율·수입물가** 계열 지표를 우선 대상으로 삼는다.

구성:
1. DART 공시에서 매출액(전체) 추출
2. 누적치를 분기 단위로 역산
3. DB에서 관련 경제지표 로드
4. 레벨 / 변화율 / 시차(lag) 세 가지 각도로 상관관계 계산

> **주의**: 표본이 20여 개 분기로 작다. 상관계수는 확정된 관계가 아니라 참고용 신호로 해석할 것.
> 공통 계산/시각화 함수는 `eda_utils.py`(같은 폴더)에 있다.

## 0. 환경 설정

In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

sys.path.append(str(Path("../../RAG").resolve()))  # dart_parser.py가 있는 폴더
import dart_parser
import eda_utils  # 계열사 노트북 공통 로직 (같은 mandu/Eda 폴더)

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "Malgun Gothic"  # Windows 기본 한글 폰트 (다른 OS라면 폰트명 교체)

In [ ]:
DART_DIR = Path("../../RAG/data/dart_xml")
TARGET_COMPANIES = {"(주)삼양패키징", "삼양패키징"}  # 표기 변형

# 지표 DB(DATABASE_URL)는 Steam_Sales/dashboard/backend/.env 에 있다
DASHBOARD_ENV = Path("../../dashboard/backend/.env")
load_dotenv(DASHBOARD_ENV)

## 1. DART 공시에서 매출액 추출

삼양사(식품)와 달리 삼양패키징은 별도 법인이라 사업부문 행이 아니라, 사업/반기/분기보고서의
"III. 재무에 관한 사항 > 1. 요약재무정보" 표에 있는 **회사 전체 매출액** 행을 뽑는다.
이 값도 그 보고서의 회계연도 시작일부터 보고 기준일까지의 누적 매출이다.

In [ ]:
REVENUE_RE = re.compile(r"매출액\s*\|\s*([\d,]+)")

cum_df, failed = eda_utils.extract_cumulative_metric(
    DART_DIR, TARGET_COMPANIES, REVENUE_RE, value_col="revenue_cum",
    section_contains="요약재무정보",  # 표 형식이 고정된 섹션으로 좁혀서 오탐 방지
)
cum_df

## 2. 누적 매출을 분기 단위로 역산

In [ ]:
revenue_df = eda_utils.cumulative_to_quarterly(cum_df, cum_col="revenue_cum", out_col="revenue")
print(f"{len(revenue_df)}개 구간 (단위: 백만원)")
revenue_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(revenue_df.index, revenue_df["revenue"], marker="o", color="#2E6FB8")
ax.fill_between(revenue_df.index, revenue_df["revenue"], revenue_df["revenue"].min() * 0.95, alpha=0.12, color="#2E6FB8")
ax.set_title("Samyang Packaging - Quarterly Revenue (KRW million)")
ax.set_ylabel("KRW million")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

## 3. DB에서 경제지표 로드

PET용기 원재료(석유화학 수지)와 관련된 유가/환율/수입물가 지표를 우선 본다.
(철강·목재 지표는 이 회사 제품군과 관련이 낮아 제외했다 — 필요하면 `TARGETS`에 추가)

In [ ]:
from sqlalchemy import create_engine
import os

engine = create_engine(os.environ["DATABASE_URL"], pool_pre_ping=True)

TARGETS = {
    "market_yfinance": ["WTI유가", "브렌트유", "천연가스", "원달러환율", "원유로환율",
                        "달러인덱스", "미국국채10년", "구리(전기동)"],
    "import_price_index": ["한국", "미국"],
    "samyang_stock_prices": ["삼양패키징"],  # 자사 주가 - 매출보다 먼저/나중에 반응하는지 확인용
}

all_series = eda_utils.load_indicator_series(engine, TARGETS)
print(f"{len(all_series)}개 지표 시계열 로드")

## 4. 기간별 지표 평균과 매출 정렬

In [ ]:
aligned = eda_utils.align_indicators_to_periods(revenue_df, "revenue", all_series)
aligned.head()

## 5. 레벨 기준 상관관계

In [ ]:
level_corr = eda_utils.corr_table(aligned, all_series, "revenue")
level_corr

In [ ]:
eda_utils.plot_top_correlations(level_corr, "Level correlation with packaging revenue (top by |r|)")

## 6. 전기 대비 변화율(%) 기준 상관관계

In [ ]:
pct = aligned.drop(columns=["period_from"]).pct_change().dropna(how="all")
pct_corr = eda_utils.corr_table(pct, all_series, "revenue", min_n=6)
pct_corr

## 7. 시차(Lag) 분석

> **다중비교 주의**: 지표 수 × lag(0~3) 조합을 모두 훑어서 최댓값을 고르는 방식이라,
> 표본이 적은 상황에서는 우연히 강한 값이 나올 수 있다. 여기서 나온 결과는
> 확정된 관계가 아니라 **검증해볼 가설 후보**로 취급할 것.

In [ ]:
level_df = aligned.drop(columns=["period_from"])
lag_df = eda_utils.lag_correlation_table(level_df, all_series, "revenue")
lag_df

In [ ]:
eda_utils.plot_lag_heatmap(lag_df, "Lag correlation (indicator at t-L vs revenue at t)", top_n=len(lag_df))

## 결론 및 한계

*(노트북을 실행한 뒤, 위 상관관계 표를 보고 이 셀에 실제 결론을 채워 넣을 것)*

- 표본이 20여 개 분기로 작다. 상관계수는 확정된 관계가 아니라 참고용 신호로 해석할 것.
- lag 분석은 지표 수 × lag 조합을 모두 탐색해 최댓값을 고르는 방식이라 다중비교 문제가 있다.